In [7]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd
import time

In [8]:
class SpotifyCollector:
    def __init__(self, client_id, client_secret):
        auth_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
        self.sp = spotipy.Spotify(auth_manager=auth_manager, retries=5, status_retries=5, backoff_factor=0.5)

    def fetch_crossmedia_tracks(self):
        print("추출 시작!")

        genre_queries = {
            '코미디/힐링': ['genre:k-indie', '어쿠스틱', '위로', '밤하늘'],
            '로맨스': ['사랑', '이별', '고백', '설렘'],
            '액션': ['genre:k-rap', '힙합', '돈', '거친'],
            '판타지': ['전설', '마법', '영웅', '에픽'],
            '스릴러/범죄': ['비밀', '거짓말', '그림자', '미스터리'],
            '공포/호러': ['악몽', '괴물', '어둠', '피'],
            '음악/스포츠': ['genre:k-pop', '댄스', '승리', '달려'],
            '무협/사극': ['조선', '달빛', '검', '사극'],
            'SF/미래': ['우주', '별', '은하수', '미래'],
            '드라마/성장': ['genre:korean ost', '청춘', '꿈', '눈물']
        }

        track_data = {}
        seen_titles_artists = set()

        for target_genre, queries in genre_queries.items():
            for query in queries:
                for offset in range(0, 30, 10): 
                    try:
                        results = self.sp.search(q=query, type='track', limit=10, offset=offset)

                        for item in results['tracks']['items']:
                            if not item or not item.get('id'): continue

                            title = item.get('name', '').strip()
                            artist = item['artists'][0]['name'].strip() if item.get('artists') else '가수 불명'

                            if not title or title == '제목 없음': continue

                            unique_key = (title.lower(), artist.lower())
                            if unique_key in seen_titles_artists:
                                continue

                            seen_titles_artists.add(unique_key)

                            thumbnail = item['album']['images'][0]['url'] if item.get('album') and item['album'].get('images') else ''

                            track_data[item['id']] = {
                                'track_id': item['id'],
                                'title': title,
                                'artist': artist,
                                'genres': target_genre, 
                                'spotify_link': item['external_urls'].get('spotify', f"https://open.spotify.com/track/{item['id']}"),
                                'thumbnail_url': thumbnail
                            }
                    except Exception:
                        continue
                    time.sleep(0.5)

        print(f"총 {len(track_data)}개의 곡 확보 완료!")
        return pd.DataFrame(list(track_data.values()))

In [ ]:
if __name__ == "__main__":
    SPOTIPY_CLIENT_ID = "" # API 키 입력
    SPOTIPY_CLIENT_SECRET = "" # API 키 입력

    collector = SpotifyCollector(client_id=SPOTIPY_CLIENT_ID, client_secret=SPOTIPY_CLIENT_SECRET)
    df_spotify = collector.fetch_crossmedia_tracks()

    if not df_spotify.empty:
        trash_keywords = ['카페', 'bgm', '태교', '수면', '명상', '자장가', 'asmr', 'inst', 'mr', '연주곡', 'instrumental', 'karaoke', '오르골', 'cover', '커버', '피아노']
        pattern = '|'.join(trash_keywords)
        df_spotify = df_spotify[~df_spotify['title'].str.contains(pattern, case=False, na=False)]

        print(f"\n 최종 정제 완료: 정제 이후 음악 총 {len(df_spotify)}개 추출 성공!")
        print(df_spotify.head(10))

        csv_filename = "spotify_music_data.csv"
        df_spotify.to_csv(csv_filename, index=False, encoding="utf-8-sig")
    else:
        print("데이터를 가져오지 못했습니다.")

추출 시작!
총 1087개의 곡 확보 완료!

 최종 정제 완료: 정제 이후 음악 총 1069개 추출 성공!
                 track_id                title             artist  genres  \
0  4ZQHxaOh3hALx6fXuRpqtv         KISS FOR YOU       Mamalaid Rag  코미디/힐링   
1  370QRRFHtg1fjSpVX5UHYv         I'm Dreamin'       Mamalaid Rag  코미디/힐링   
2  0EUPCSi9xOiD8izVOVq5T5               Mexico       Destiny Road  코미디/힐링   
3  0K0Ri913PLEK6LlzRMRiBQ          Ocean Ocean         The Chills  코미디/힐링   
4  5YeWcY6ZJLiwurGayztmZu            한번 더 생각해봐   PAPERCUT PROJECT  코미디/힐링   
5  5Hbvy2ptEsXtTPuCAhRdj3          Lost Myself       Jo MacKenzie  코미디/힐링   
6  5GZTpTolPHO6hxqyxerVih     We'll Never Know  Jon Whitlock Trio  코미디/힐링   
7  48sYnuPGcEbWQJ2PVOebAX  La Fazan, ku je ti?          La Fazani  코미디/힐링   
8  29br4h300e5GpOxrl74diU             66 Knots      Oakridge Ave.  코미디/힐링   
9  4WLBjXfvqQ2Cm8Ivoi7AWm     Get What You Get    Bears and Lions  코미디/힐링   

                                        spotify_link  \
0  https://open.spotify.com/track/4